## Requirements

In [ ]:
%pip install wfdb --no-deps

## Import and Configuration

In [ ]:
# Standard Library
import os
import random

# Data Manipulation & Math
import numpy as np
import pandas as pd
from tqdm import tqdm

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Signal Processing & Domain Specific
import wfdb
from scipy.signal import butter, filtfilt

# Scikit-Learn (Machine Learning)
from sklearn.metrics import (
    f1_score, roc_auc_score, confusion_matrix, 
    roc_curve, precision_recall_curve
)

# TensorFlow / Keras (Deep Learning)
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import AUC
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam, schedules
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, MaxPooling1D, 
    Dense, Reshape, Dropout, Concatenate, GlobalAveragePooling1D,
    Flatten, SpatialDropout1D, Add, Lambda, Dense, Multiply, LSTM,
    Bidirectional, GaussianNoise, MultiHeadAttention, LayerNormalization
)

# Configuration & Constants
INPUT_DIR = "/kaggle/input/"
OUTPUT_DIR = "/kaggle/working/"
DATASET_PATH = os.path.join(INPUT_DIR, "datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1")
FILE_PTB = os.path.join(DATASET_PATH, "ptbxl_database.csv")
FILE_SCP = os.path.join(DATASET_PATH, "scp_statements.csv")

SEED = 42
SAMPLE_RATE = 500
CLASS_NAMES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
CLASS_NUM = len(CLASS_NAMES)
MAX_RECORDS = 10000

INPUT_SHAPE = (SAMPLE_RATE * 10, 12)
EPOCHS = 150
BATCH_SIZE = 16

# Set Seeds
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Helper Functions

In [ ]:
def safe_eval_scp(scp_code):
    try: 
        return eval(scp_code) if isinstance(scp_code, str) else scp_code
    except: 
        return {}

def apply_bandpass_filter(signal, fs, lowcut=0.5, highcut=45.0, order=2):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='bandpass')
    return filtfilt(b, a, signal, axis=0)

def ensure_len(signal, target_len):
    current_len = len(signal)
    if current_len > target_len:
        return signal[:target_len]
    elif current_len < target_len:
        pad_len = target_len - current_len
        return np.pad(signal, ((0, pad_len), (0, 0)), mode='constant')
    return signal

def normalize_dataset(X):
    mean = np.mean(X, axis=1, keepdims=True)
    std = np.std(X, axis=1, keepdims=True)
    std[std == 0] = 1.0
    return (X - mean) / std

def find_optimal_thresholds(y_true, y_prob, class_names=CLASS_NAMES):
    opt_thresholds = []
    thresholds = np.linspace(0.05, 0.8, 100)

    for i in range(len(class_names)):
        f1_scores = [f1_score(y_true[:, i], (y_prob[:, i] >= t).astype(int), zero_division=0) 
                     for t in thresholds]
        
        best_idx = np.argmax(f1_scores)
        best_t = thresholds[best_idx]
        best_f1 = f1_scores[best_idx]
        
        opt_thresholds.append(best_t)
        print(f"Class: {class_names[i]:<6} | Best T: {best_t:.3f} | F1: {best_f1:.4f}")
        
    return opt_thresholds


## Load Metadata

In [ ]:
def load_metadata(file_ptb=FILE_PTB, file_scp=FILE_SCP, class_names=CLASS_NAMES, max_records=MAX_RECORDS):
    # Load raw dataframes
    ptb_df = pd.read_csv(file_ptb, index_col='ecg_id')
    scp_df = pd.read_csv(file_scp, index_col=0)

    # Convert scp_codes from string to dict
    ptb_df["scp_codes"] = ptb_df["scp_codes"].apply(safe_eval_scp)
    
    # Filter for records with valid filenames (using 100Hz version)
    ptb_df = ptb_df[ptb_df["filename_lr"].notna()] 

    # Build the Dictionary Map (SCP Code -> Diagnostic Class)
    scp_df = scp_df[scp_df["diagnostic_class"].notna()]
    diag_class = scp_df["diagnostic_class"].to_dict()

    # Define Multi-label Vectorization Function
    def get_multilabel_vector(scp_dict):
        vec = np.zeros(len(class_names), dtype=np.float32)
        for scp_code in scp_dict.keys():
            if scp_code in diag_class:
                cls = diag_class[scp_code]
                if cls in class_names:
                    vec[class_names.index(cls)] = 1.0
        return vec

    # Apply Multi-label Vectors
    ptb_df["label_vector"] = ptb_df["scp_codes"].apply(get_multilabel_vector)

    # Filter out records that don't belong to any of the 5 superclasses
    ptb_df = ptb_df[ptb_df["label_vector"].apply(lambda v: np.sum(v) > 0)]

    # Optional Sampling
    if max_records:
        ptb_df = ptb_df.sample(n=min(max_records, len(ptb_df)), random_state=SEED)

    return ptb_df.reset_index()

ptb_df = load_metadata()
print("Total valid records: ", {len(ptb_df)})
ptb_df[["ecg_id", "scp_codes", "label_vector"]]

## Load One Signal and Smoke Test

In [ ]:
def load_signal_single(filename, base_path=DATASET_PATH):
    try:
        path = os.path.join(base_path, filename)
        signal, _ = wfdb.rdsamp(path)
        return signal.astype(np.float32)
    except:
        return None

def smoke_test(df, sample_rate=SAMPLE_RATE, count=1):
    print(f"Showing first {count} records (12-Lead view)...")

    lead_names = ['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

    for row in df.itertuples():
        file = "filename_lr" if sample_rate == 100 else "filename_hr"
        sig = load_signal_single(getattr(row, file))
        if sig is None:
            continue

        # Ensure length
        sig = ensure_len(sig, sample_rate * 10)

        # Plot all 12 leads stacked
        fig, axes = plt.subplots(12, 1, figsize=(10, 15), sharex=True)

        for i in range(12):
            ax = axes[i]
            ax.plot(sig[:, i], color='black', linewidth=0.8)
            ax.set_ylabel(lead_names[i], rotation=0, labelpad=20, fontsize=10, fontweight='bold')
            ax.grid(True, linestyle='--', alpha=0.5)
            if i < 11:
                ax.tick_params(labelbottom=False)

        plt.suptitle(f"Record: {getattr(row, file)}\nMulti: {row.label_vector}", y=1.02)
        plt.xlabel("Time (samples)")
        plt.tight_layout()
        plt.show()

        count -= 1
        if count == 0:
            return

smoke_test(ptb_df)

## Build and Split Dataset

In [ ]:
def build_dataset(df, sample_rate=SAMPLE_RATE):
    X, y, folds = [], [], []
    file = "filename_lr" if sample_rate == 100 else "filename_hr"

    for row in tqdm(df.itertuples(), total=len(df), desc=f"Loading {sample_rate}Hz ECGs"):
        sig = load_signal_single(getattr(row, file)) 
        if sig is None: continue
            
        # Apply Clinical Bandpass Filter
        sig = apply_bandpass_filter(sig, fs=float(sample_rate))
        
        # Standardize length to 10 seconds (100 * 10 or 500 * 10)
        sig = ensure_len(sig, sample_rate * 10) 

        X.append(sig)
        y.append(row.label_vector)
        folds.append(row.strat_fold)

    # Stack to create 3D Array: (records, samples, 12)
    X = np.stack(X)
    y = np.stack(y)
    folds = np.array(folds)
    
    return X, y, folds

# Build
X, y, folds = build_dataset(ptb_df)

# Normalization
X = normalize_dataset(X)

# Split (1-8: Train, 9: Val, 10: Test)
train_idx = np.where(folds <= 8)[0]
val_idx   = np.where(folds == 9)[0]
test_idx  = np.where(folds == 10)[0]

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

# Display Shapes 
print("\n--- Dataset Description ---")
print(f"Total Records: {len(X)}")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape} | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

## Model Block Defination and Universal Evaluator

In [ ]:
def dense_block(x, filters, kernel_size, padding='same', activation='swish'):
    # Save the Features
    shortcut = x
    
    # Feature Extraction Path
    x = Conv1D(filters, kernel_size, padding=padding, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    
    x = SpatialDropout1D(0.2)(x)
    
    x = Conv1D(filters, kernel_size, padding=padding, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    
    # Dense Connection: Concatenate input with extracted features
    x = Concatenate()([shortcut, x]) 
    x = Activation(activation)(x)
    
    return x

def residual_block(x, filters, kernel_size, padding='same', activation='swish'):
    # Save the Features
    shortcut = x
    
    # Feature Extraction Path
    x = Conv1D(filters, kernel_size, padding=padding, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    
    x = SpatialDropout1D(0.2)(x)
    
    x = Conv1D(filters, kernel_size, padding=padding, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    
    # Dimension Matching to Current x
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Residual Connection: Add input with extracted features
    x = Add()([x, shortcut]) 
    x = Activation(activation)(x)
    
    return x

def attention_block(x, num_heads=4, key_dim=128, dropout=0.1):
    # Self-Attention: x acts as Query, Key, and Value
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=dropout)(x, x)
    
    # Residual Connection + Layer Normalization
    x = Add()([x, attn])
    x = LayerNormalization(epsilon=1e-6)(x)
    
    return x

def evaluate_model(model_name, y_true_bin, y_pred_prob, thresholds, class_names=CLASS_NAMES): 
    # Apply threshold to get binary predictions for F1 and Confusion Matrix
    y_pred_bin = np.zeros_like(y_pred_prob)
    for i in range(len(class_names)):
        y_pred_bin[:, i] = (y_pred_prob[:, i] >= thresholds[i]).astype(int)

    # --- Metrics Summary Table ---
    metrics_data = []
    for i in range(len(class_names)):
        c_auc = roc_auc_score(y_true_bin[:, i], y_pred_prob[:, i])
        c_f1 = f1_score(y_true_bin[:, i], y_pred_bin[:, i])
        metrics_data.append({"Class": class_names[i], "AUC-ROC": c_auc, "F1-Score": c_f1})
    
    # Calculate Macro Averages
    macro_auc = roc_auc_score(y_true_bin, y_pred_prob, average='macro')
    macro_f1 = f1_score(y_true_bin, y_pred_bin, average='macro')
    
    df_metrics = pd.DataFrame(metrics_data)
    summary_row = pd.DataFrame([{"Class": "MACRO-AVG", "AUC-ROC": macro_auc, "F1-Score": macro_f1}])
    df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)
    
    print(f"\n{'='*60}")
    print(f"EVALUATION: {model_name}")
    print(f"{'='*60}")
    print(df_metrics.to_string(index=False))
    print("\n")

    # --- Confusion Matrices ---
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    for i in range(len(class_names)):
        cm = confusion_matrix(y_true_bin[:, i], y_pred_bin[:, i])
        sns.heatmap(
            cm, annot=True, fmt='d', ax=axes[i], cmap='Blues', cbar=False,
            square=True, annot_kws={"size": 20, "weight": "bold"}
        )
        axes[i].tick_params(axis='both', which='major', labelsize=16)
        axes[i].set_title(f'CM: {class_names[i]}', fontsize=20, fontweight='bold')
        axes[i].set_xlabel('Predicted', fontsize=18)
        axes[i].set_ylabel('Actual', fontsize=18)
        
    plt.tight_layout()
    plt.show()

    # --- ROC and PR Curves ---
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Calculate and plot per-class AUC-ROC
    fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
    for i in range(len(class_names)):
        fpr_dict[i], tpr_dict[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        axes[0].plot(fpr_dict[i], tpr_dict[i], lw=1.5, label=f'ROC {class_names[i]}')

    # Calculate Macro-average ROC (mean of all FPR/TPR)
    all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(len(class_names))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(class_names)):
        mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
    mean_tpr /= len(class_names)
    
    axes[0].plot(all_fpr, mean_tpr, label=f'MACRO-AVG', color='black', linestyle=':', linewidth=4)
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate', fontsize=12)
    axes[0].set_ylabel('True Positive Rate', fontsize=12)
    axes[0].set_title('ROC Curves', fontsize=15)
    axes[0].legend(loc="lower right", fontsize=12)
    axes[0].grid(alpha=0.3)

    # Calculate and plot per-class AUC-PR
    precision_dict, recall_dict, ap_dict = {}, {}, {}
    for i in range(len(class_names)):
        precision_dict[i], recall_dict[i], _ = precision_recall_curve(y_true_bin[:, i], y_pred_prob[:, i])
        axes[1].plot(recall_dict[i], precision_dict[i], lw=1.5, label=f'PR {class_names[i]}')

    axes[1].set_xlabel('Recall', fontsize=12)
    axes[1].set_ylabel('Precision', fontsize=12)
    axes[1].set_title(f'PR Curves', fontsize=15)
    axes[1].legend(loc="upper right", fontsize=12)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## DenseNet Model Building and Training

In [ ]:
# --- Define the Model ---
def build_dense_net(input_shape=INPUT_SHAPE, class_num=CLASS_NUM):
    # Input Layer
    inputs = Input(shape=input_shape)
    
    # Noise Layer
    inputs = GaussianNoise(0.01)(inputs)
    
    # Dense Layer
    x = dense_block(inputs, filters=36, kernel_size=5)
    x = MaxPooling1D(pool_size=2)(x) 

    # Dense Layer
    x = dense_block(x, filters=72, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Dense Layer
    x = dense_block(x, filters=144, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Dense Layer
    x = dense_block(x, filters=216, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Dense Layer
    x = dense_block(x, filters=288, kernel_size=3) 
    x = MaxPooling1D(pool_size=2)(x)  
    
    # Dense and GAP Layer
    x = dense_block(x, filters=288, kernel_size=3) 
    x = GlobalAveragePooling1D()(x)
    
    # Bottleneck Layer
    x = Dense(128, activation='swish', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    outputs = Dense(class_num, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="One_Branch_DenseNet")
    return model

# --- Compile and Train the Model ---
dense_net = build_dense_net()

steps_per_epoch = len(X_train) // BATCH_SIZE
lr_schedule = CosineDecay(
    initial_learning_rate=0.0005, 
    decay_steps=EPOCHS * steps_per_epoch,
    alpha=0.01
)

dense_net.compile(
    optimizer=Adam(learning_rate=lr_schedule), 
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=[AUC(name='auc', multi_label=True)]
)

# dense_net.summary()

callbacks_dense_net = [
    EarlyStopping(
        monitor='val_auc', mode='max', patience=20, 
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "DenseNet.keras"), 
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    )
]

history_dense_net = dense_net.fit(
    X_train, y_train,        # Folds 1-8
    validation_data=(X_val, y_val), # Fold 9
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_dense_net,
    verbose=1
)

## DenseNet Model Evaluation

In [ ]:
# Predict on Validation and Test Sets
print("Generating Probabilities...")
y_val_prob = dense_net.predict(X_val, verbose=0)
y_test_prob = dense_net.predict(X_test, verbose=0)

# Find Optimal Threshold on Validation Set (Fold 9)
print("\nTuning threshold on Validation Set...")
opt_thresholds = find_optimal_thresholds(y_val, y_val_prob)

# Evaluate on Test Set (Fold 10) using tuned threshold
print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
res_dense_net = evaluate_model("Dense Net", y_test, y_test_prob, opt_thresholds)

## ResNet Model Building and Training

In [ ]:
# --- Define the Model ---
def build_res_net(input_shape=INPUT_SHAPE, class_num=CLASS_NUM):
    # Input Layer
    inputs = Input(shape=input_shape)
    
    # Noise Layer
    inputs = GaussianNoise(0.01)(inputs)
    
    # Residual Layer
    x = residual_block(inputs, filters=36, kernel_size=5)
    x = MaxPooling1D(pool_size=2)(x) 

    # Residual Layer
    x = residual_block(x, filters=72, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual Layer
    x = residual_block(x, filters=144, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual Layer
    x = residual_block(x, filters=216, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual and GAP Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = GlobalAveragePooling1D()(x)
    
    # Bottleneck Layer
    x = Dense(128, activation='swish', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    outputs = Dense(class_num, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="One_Branch_ResNet")
    return model

# --- Compile and Train the Model ---
res_net = build_res_net()

steps_per_epoch = len(X_train) // BATCH_SIZE
lr_schedule = CosineDecay(
    initial_learning_rate=0.0005, 
    decay_steps=EPOCHS * steps_per_epoch,
    alpha=0.01
)

res_net.compile(
    optimizer=Adam(learning_rate=lr_schedule), 
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=[AUC(name='auc', multi_label=True)]
)

# res_net.summary()

callbacks_res_net = [
    EarlyStopping(
        monitor='val_auc', mode='max', patience=20, 
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "ResNet.keras"), 
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    )
]

history_res_net = res_net.fit(
    X_train, y_train,        # Folds 1-8
    validation_data=(X_val, y_val), # Fold 9
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_res_net,
    verbose=1
)

## ResNet Model Evaluation

In [ ]:
# Predict on Validation and Test Sets
print("Generating Probabilities...")
y_val_prob = res_net.predict(X_val, verbose=0)
y_test_prob = res_net.predict(X_test, verbose=0)

# Find Optimal Threshold on Validation Set (Fold 9)
print("\nTuning threshold on Validation Set...")
opt_thresholds = find_optimal_thresholds(y_val, y_val_prob)

# Evaluate on Test Set (Fold 10) using tuned threshold
print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
res_res_net = evaluate_model("Res Net", y_test, y_test_prob, opt_thresholds)

## AG-ResNet Model Building and Training

In [ ]:
# --- Define the Model ---
def build_ag_res_net(input_shape=INPUT_SHAPE, class_num=CLASS_NUM):
    # Input Layer
    inputs = Input(shape=input_shape)
    
    # Noise Layer
    inputs = GaussianNoise(0.01)(inputs)

    # Branching into 5 Branch
    septal = Lambda(lambda x: tf.gather(x, [6, 7], axis=2), name="Septal")(inputs)
    anterior = Lambda(lambda x: tf.gather(x, [8, 9], axis=2), name="Anterior")(inputs)
    lateral = Lambda(lambda x: tf.gather(x, [0, 4, 10, 11], axis=2), name="Lateral")(inputs)
    inferior = Lambda(lambda x: tf.gather(x, [1, 2, 5], axis=2), name="Inferior")(inputs)
    posterior = Lambda(lambda x: tf.gather(x, [3], axis=2), name="Posterior")(inputs) 

    branches_in = [septal, anterior, lateral, inferior, posterior]
    branches_out = []

    for b in branches_in:
        # Residual Layer
        x = residual_block(b, filters=36, kernel_size=5)
        x = MaxPooling1D(pool_size=2)(x)

        # Residual Layer
        x = residual_block(x, filters=72, kernel_size=3)
        x = MaxPooling1D(pool_size=2)(x) 
        
        # Residual Layer
        x = residual_block(x, filters=144, kernel_size=3)
        x = MaxPooling1D(pool_size=2)(x) 

        branches_out.append(x)

    x = Concatenate()(branches_out)
    
    # Residual Layer
    x = residual_block(x, filters=216, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual and GAP Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = GlobalAveragePooling1D()(x)
    
    # Bottleneck Layer
    x = Dense(128, activation='swish', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    outputs = Dense(class_num, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="Five_Branch_AG_ResNet")
    return model

# --- Compile and Train the Model ---
ag_res_net = build_ag_res_net()

steps_per_epoch = len(X_train) // BATCH_SIZE
lr_schedule = CosineDecay(
    initial_learning_rate=0.0005, 
    decay_steps=EPOCHS * steps_per_epoch,
    alpha=0.01
)

ag_res_net.compile(
    optimizer=Adam(learning_rate=lr_schedule), 
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=[AUC(name='auc', multi_label=True)]
)

# ag_res_net.summary()

callbacks_ag_res_net = [
    EarlyStopping(
        monitor='val_auc', mode='max', patience=20, 
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "AG-ResNet.keras"), 
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    )
]

history_ag_res_net = ag_res_net.fit(
    X_train, y_train,        # Folds 1-8
    validation_data=(X_val, y_val), # Fold 9
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_ag_res_net,
    verbose=1
)

## AG-ResNet Model Evaluation

In [ ]:
# Predict on Validation and Test Sets
print("Generating Probabilities...")
y_val_prob = ag_res_net.predict(X_val, verbose=0)
y_test_prob = ag_res_net.predict(X_test, verbose=0)

# Find Optimal Threshold on Validation Set (Fold 9)
print("\nTuning threshold on Validation Set...")
opt_thresholds = find_optimal_thresholds(y_val, y_val_prob)

# Evaluate on Test Set (Fold 10) using tuned threshold
print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
res_ag_res_net = evaluate_model("AG-ResNet", y_test, y_test_prob, opt_thresholds)

## AGGC-ResNet Model Building and Training

In [ ]:
# --- Define the Model ---
def build_aggc_res_net(input_shape=INPUT_SHAPE, class_num=CLASS_NUM):
    # Input Layer
    inputs = Input(shape=input_shape)
    
    # Noise Layer
    inputs = GaussianNoise(0.01)(inputs)

    # Branching into 5 Branch
    septal = Lambda(lambda x: tf.gather(x, [6, 7], axis=2), name="Septal")(inputs)
    anterior = Lambda(lambda x: tf.gather(x, [8, 9], axis=2), name="Anterior")(inputs)
    lateral = Lambda(lambda x: tf.gather(x, [0, 4, 10, 11], axis=2), name="Lateral")(inputs)
    inferior = Lambda(lambda x: tf.gather(x, [1, 2, 5], axis=2), name="Inferior")(inputs)
    posterior = Lambda(lambda x: tf.gather(x, [3], axis=2), name="Posterior")(inputs) 

    branches_in = [
        (inputs, 31, 15), (septal, 5, 3), (anterior, 5, 3), 
        (lateral, 7, 5), (inferior, 7, 5), (posterior, 3, 3)
    ]
    branches_out = []

    for b, k0, k1 in branches_in:
        # Residual Layer
        x = residual_block(b, filters=36, kernel_size=k0)
        x = MaxPooling1D(pool_size=2)(x)

        # Residual Layer
        x = residual_block(x, filters=72, kernel_size=k1)
        x = MaxPooling1D(pool_size=2)(x) 
        
        # Residual Layer
        x = residual_block(x, filters=144, kernel_size=k1)
        x = MaxPooling1D(pool_size=2)(x) 

        branches_out.append(x)

    # Fusion and Attention Layer
    x = Concatenate()(branches_out)
    x = attention_block(x, num_heads=4, key_dim=x.shape[-1])
    
    # Residual Layer
    x = residual_block(x, filters=216, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = MaxPooling1D(pool_size=2)(x) 
    
    # Residual and BI-LSTM Layer
    x = residual_block(x, filters=288, kernel_size=3)
    x = Bidirectional(LSTM(128, return_sequences=False))(x)
    
    # Bottleneck Layer
    x = Dense(128, activation='swish', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    # Output Layer
    outputs = Dense(class_num, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="Six_Branch_AGGC_ResNet")
    return model

# --- Compile and Train the Model ---
aggc_res_net = build_aggc_res_net()

steps_per_epoch = len(X_train) // BATCH_SIZE
lr_schedule = CosineDecay(
    initial_learning_rate=0.0005, 
    decay_steps=EPOCHS * steps_per_epoch,
    alpha=0.01
)

aggc_res_net.compile(
    optimizer=Adam(learning_rate=lr_schedule), 
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=[AUC(name='auc', multi_label=True)]
)

# aggc_res_net.summary()

callbacks_aggc_res_net = [
    EarlyStopping(
        monitor='val_auc', mode='max', patience=20, 
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, "AGGC-ResNet.keras"), 
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    )
]

history_aggc_res_net = aggc_res_net.fit(
    X_train, y_train,        # Folds 1-8
    validation_data=(X_val, y_val), # Fold 9
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_aggc_res_net,
    verbose=1
)

## AGGC-ResNet Model Evaluation

In [ ]:
# Predict on Validation and Test Sets
print("Generating Probabilities...")
y_val_prob = aggc_res_net.predict(X_val, verbose=0)
y_test_prob = aggc_res_net.predict(X_test, verbose=0)

# Find Optimal Threshold on Validation Set (Fold 9)
print("\nTuning threshold on Validation Set...")
opt_thresholds = find_optimal_thresholds(y_val, y_val_prob)

# Evaluate on Test Set (Fold 10) using tuned threshold
print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
res_ag_res_net = evaluate_model("AGGC-ResNet", y_test, y_test_prob, opt_thresholds)

## CNN Model Building and Training (Binary Classification)

In [ ]:
# # Configuration
# INPUT_SHAPE = (SAMPLE_RATE * 10, 12)
# EPOCHS = 50
# BATCH_SIZE = 32

# # Define CNN Architecture
# def build_cnn(input_shape=INPUT_SHAPE):
#     """
#     Description: Constructs a 1D CNN with hierarchical blocks for multi-label 
#                  diagnostic classification using Sigmoid output and Macro-AUC.
#     Input: tuple (samples, leads)
#     Output: keras.Model
#     """
#     model = Sequential()
#     model.add(Input(shape=input_shape))                              # Shape: (32, 1000, 12)

#     # Block 1: Low-level features
#     model.add(Conv1D(filters=32, kernel_size=32, strides=2, padding='same'))    # Shape: (32, 500, 32)
#     model.add(BatchNormalization())
#     model.add(Activation('relu'))

#     # Block 2: Mid-level features
#     model.add(Conv1D(filters=64, kernel_size=15, strides=2, padding='same'))    # Shape: (32, 250, 64)
#     model.add(BatchNormalization())
#     model.add(Activation('relu'))

#     # Block 3: High-level features
#     model.add(Conv1D(filters=128, kernel_size=11, padding='same'))   # Shape: (32, 250, 128)
#     model.add(BatchNormalization())
#     model.add(Activation('relu'))

#     # Classifier Head
#     model.add(GlobalAveragePooling1D())                              # Shape: (32, 128)
#     model.add(Dense(128, activation='relu'))                         # Shape: (32, 128)
#     model.add(Dropout(0.5))
    
#     # Multi-label Output: 5 nodes with independent probabilities
#     model.add(Dense(NUM_CLASSES, activation='sigmoid'))              # Shape: (32, 5)

#     # Compile with Research Gold Standard (Macro-AUC)
#     model.compile(
#         optimizer=Adam(learning_rate=0.001), 
#         loss='binary_crossentropy', 
#         metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
#     )
#     return model

# # Build the model
# print(f"\n{'='*40}")
# print("BUILDING RESEARCH CNN")
# print(f"{'='*40}")
# cnn = build_cnn()
# cnn.summary()

# # Train using the Splits (Folds 1-8 for train, 9 for val)
# callbacks_cnn = [
#     EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True, verbose=1),
#     ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=5, min_lr=0.00001, verbose=1)
# ]

# history_cnn = cnn.fit(
#     X_train, y_train,        # Folds 1-8
#     validation_data=(X_val, y_val), # Fold 9
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     callbacks=callbacks_cnn,
#     verbose=1
# )

# # Save the model
# cnn.save(os.path.join(OUTPUT_DIR, 'cnn_model.keras'))

##  CNN Model Evaluation (Binary Classification)

In [ ]:
# # Predict on Validation and Test Sets
# print("Generating Probabilities...")
# y_prob_val = cnn.predict(X_val, verbose=0)
# y_prob_test = cnn.predict(X_test, verbose=0)

# # Find Optimal Threshold on Validation Set (Fold 9)
# print("\nTuning threshold on Validation Set...")
# opt_threshold = find_optimal_thresholds(y_val, y_prob_val)

# # Evaluate on Test Set (Fold 10) using tuned threshold
# print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
# res_cnn = evaluate_model("1D CNN", y_test, y_prob_test, opt_threshold)

## AG-ResNET Model Building and Training (Binary Classification)

In [ ]:
# # Configuration
# INPUT_SHAPE = (SAMPLE_RATE * 10, 12)
# NUM_CLASSES = 5
# NUM_BLOCKS = 3
# FILTERS_START = 32

# LEARNING_RATE = 0.0005 
# BATCH_SIZE = 16
# EPOCHS = 50

# # Define Residual Block Architecture
# def residual_block(x, filters, kernel_size=10, stride=1):
#     """
#     Description: Implements a standard 1D Residual bottleneck with 'swish' activation and 
#                  spatial dropout; includes a 1x1 convolution shortcut if dimensions change.
#     Input: tensorflow.Tensor [time, channels], int, int, int
#     Output: tensorflow.Tensor [time/stride, filters]
#     """

#     shortcut = x

#     # Path A
#     x = Conv1D(filters, kernel_size, strides=stride, padding='same')(x)
#     x = BatchNormalization()(x)
#     x = Activation('swish')(x)
#     x = SpatialDropout1D(0.2)(x)

#     x = Conv1D(filters, kernel_size, strides=1, padding='same')(x)
#     x = BatchNormalization()(x)

#     # Path B
#     if x.shape[-1] != shortcut.shape[-1] or stride != 1:
#         shortcut = Conv1D(filters, 1, strides=stride, padding='same')(shortcut)
#         shortcut = BatchNormalization()(shortcut)

#     x = Add()([x, shortcut])
#     x = Activation('swish')(x)
#     return x

# # Define AG ResNet Architecture
# def build_ag_resnet(input_shape=INPUT_SHAPE):
#     """
#     Description: Constructs the Anatomically Grouped ResNet by splitting 12 leads into 
#                  5 regional branches (Septal, Anterior, Lateral, Inferior, Posterior) before 
#                  fusing them for classification.
#     Note: samples = signal duration * frequency
#     Input: tuple (samples, leads)
#     Output: keras.src.models.model.Model
#     """
#     # --- BLOCK 1: INPUT & TRIAGE ---
#     input_layer = Input(shape=input_shape, name="ecg_input") # Shape: (16, 5000, 12)
 
#     septal = Lambda(lambda x: __import__('tensorflow').gather(x, [6, 7], axis=2), name="Septal")(input_layer)            # Shape: (16, 5000, 2)
#     anterior = Lambda(lambda x: __import__('tensorflow').gather(x, [8, 9], axis=2), name="Anterior")(input_layer)        # Shape: (16, 5000, 2)
#     lateral = Lambda(lambda x: __import__('tensorflow').gather(x, [0, 4, 10, 11], axis=2), name="Lateral")(input_layer)  # Shape: (16, 5000, 4)
#     inferior = Lambda(lambda x: __import__('tensorflow').gather(x, [1, 2, 5], axis=2), name="Inferior")(input_layer)     # Shape: (16, 5000, 3)
#     posterior = Lambda(lambda x: __import__('tensorflow').gather(x, [3], axis=2), name="Posterior")(input_layer)         # Shape: (16, 5000, 1)

#     # --- BLOCK 2: ENCODER ---
#     branches = []

#     for name, branch in zip(["Global", "Septal", "Anterior", "Lateral", "Inferior", "Posterior"],
#                             [input_layer, septal, anterior, lateral, inferior, posterior]):

#         # Initial Feature Extraction
#         x = Conv1D(FILTERS_START, 32, strides=2, padding='same', name=f"{name}_Conv1")(branch) # Shape: (16, 2500, 64)
#         x = BatchNormalization()(x)                                                                   # Shape: (16, 2500, 64)
#         x = Activation('swish')(x)                                                                    # Shape: (16, 2500, 64)

#         # Dynamic Residual Stacking
#         current_filters = FILTERS_START * 2
#         for i in range(NUM_BLOCKS):
#             x = residual_block(x, current_filters, stride=2) # Shape: (16, 1250, 128) -> (16, 625, 256) -> (16, 313, 512) -> (16, 157, 1024)
#             current_filters *= 2

#         x = GlobalAveragePooling1D(name=f"{name}_GlobalPool")(x) # Shape: (16, 1024)
#         branches.append(x)                                              # Shape: (16, 6, 1024)

#     # --- BLOCK 3: FUSION ---
#     m = Concatenate(name="Anatomical_Fusion")(branches) # Shape: (16, 6 * 1024)
#     m = Reshape((len(branches), -1))(m) 
#     m = Bidirectional(LSTM(128, return_sequences=False))(m)

#     # --- BLOCK 4: CLASSIFICATION HEAD ---
#     c = BatchNormalization()(m)                         # Shape: (16, 5120)
#     c = Dense(256, activation='swish')(c)                    # Shape: (16, 128)
#     c = Dropout(0.4)(c)                                      # Shape: (16, 128)
    
#     output_layer = Dense(NUM_CLASSES, activation='sigmoid', name="classification")(c)  # Shape: (16, 1)

#     # --- BLOCK 5: COMPILATION (Pure Classification) ---
#     model = Model(inputs=input_layer, outputs=output_layer, name="AG_ResNet")

#     # Added clipnorm=1.0 to prevent gradient explosions
#     model.compile(
#         optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=1.0), 
#         loss="binary_crossentropy",
#         metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
#     )

#     return model

# # Build
# print(f"{'='*50}")
# print(f"AG-ResNet TRAINING (PURE CLASSIFIER)")
# print(f"Input: {INPUT_SHAPE}")
# print(f"Structure: {NUM_BLOCKS} Blocks | Start Filters: {FILTERS_START}")
# print(f"{'='*50}")

# ag_resnet = build_ag_resnet()

# # Train
# callbacks_ag_resnet = [
#     ReduceLROnPlateau(
#         monitor='val_auc', factor=0.2, 
#         patience=5, min_lr=1e-5, verbose=1
#     ),
#     EarlyStopping(
#         monitor='val_auc', mode='max', patience=12, 
#         restore_best_weights=True, verbose=1
#     ),
    # ModelCheckpoint(
    #     os.path.join(OUTPUT_DIR, "ag_resnet_multilabel.keras"), 
    #     monitor='val_auc', mode='max', save_best_only=True, verbose=1
    # )
# ]

# print("\nStarting Training...")
# history_ag_resnet = ag_resnet.fit(
#     x=X_train,              # Folds 1-8
#     y=y_train,              # Multi-label vectors [NORM, MI, STTC, CD, HYP]
#     validation_data=(X_val, y_val), # Fold 9
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     callbacks=callbacks_ag_resnet,
#     verbose=1
# )

## AG-ResNET Model Evaluation (Binary Classification)

In [ ]:
# # Predict on Validation and Test Sets
# print("Generating Probabilities...")
# y_prob_val = ag_resnet.predict(X_val, verbose=0)
# y_prob_test = ag_resnet.predict(X_test, verbose=0)

# # Find Optimal Threshold on Validation Set (Fold 9)
# print("\nTuning threshold on Validation Set...")
# opt_threshold = find_optimal_threshold(y_val, y_prob_val)

# # Evaluate on Test Set (Fold 10) using tuned threshold
# print("\nPerforming Final Evaluation on Test Set (Fold 10)...")
# res_ag_resnet = evaluate_model("AG-ResNet", y_test, y_prob_test, opt_threshold)